# Unitree-Go2-Flat-MethodA-Electric — 계산 흐름 노트 (compact)

한 정책 주기 안에서 (정책 20 ms → PD 5 ms → 물리 0.1 ms × 200) 의 흐름을
값으로 따라가는 노트. 모델은 직접 접촉이 없는 hip 관절만 다루는 1자유도
($J_c = 1$) 축소이고, motor 는 $K_e$ mismatch 없는 healthy 상태로 둔다.
자세한 소스 인용은 §7 참조표 참고.

## 0. 개요

- 대상: Unitree Go2 평지 보행, task ID `Unitree-Go2-Flat-MethodA-Electric`
- 시간 척도: 정책 20 ms / PD 5 ms / 물리 0.1 ms (`timestep=1e-4`, `decimation=200`,
  `_PD_RECOMPUTE=50`)
- "Method A": $\beta = 1/(1+h/\tau)$ 를 적분기·Schur·Force RHS 모두에 일관 적용
  (BE 분기).
- "Electric": 모터 전류 $I$ 를 MuJoCo activation state (`d->act`) 에 통합한
  BLDC 모델 (`dyntype = filterexact` + Schur cross-Jacobian).


## 1. 시간 척도 계층

한 정책 주기 (20 ms) 안에는 PD 가 4 회, 물리 substep 이 200 회 일어난다.
**바깥 척도가 산출한 값은 다음 갱신까지 안쪽 척도가 캐시처럼 들고 쓴다.**

<svg width="100%" viewBox="0 0 680 290" xmlns="http://www.w3.org/2000/svg" style="max-width: 720px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">
<style>
.pol-fill{fill:#EEEDFE;stroke:#534AB7}.pol-title{fill:#3C3489}.pol-sub{fill:#534AB7}
.teal-fill{fill:#E1F5EE;stroke:#0F6E56}.teal-title{fill:#085041}.teal-sub{fill:#0F6E56}
.coral-fill{fill:#FAECE7;stroke:#993C1D}.coral-title{fill:#712B13}.coral-sub{fill:#993C1D}
.heading{fill:#FFFFFF}.caption{fill:#FFFFFF}.axis{stroke:#666}
@media (prefers-color-scheme: dark){
.pol-fill{fill:#3C3489;stroke:#AFA9EC}.pol-title{fill:#CECBF6}.pol-sub{fill:#AFA9EC}
.teal-fill{fill:#085041;stroke:#5DCAA5}.teal-title{fill:#9FE1CB}.teal-sub{fill:#5DCAA5}
.coral-fill{fill:#712B13;stroke:#F0997B}.coral-title{fill:#F5C4B3}.coral-sub{fill:#F0997B}
.heading{fill:#D3D1C7}.caption{fill:#B4B2A9}.axis{stroke:#444441}
}
</style>
<text x="120" y="22" font-size="12" class="caption">t = t_k</text>
<text x="640" y="22" font-size="12" class="caption" text-anchor="end">t_k + 20 ms</text>
<line x1="120" y1="30" x2="640" y2="30" class="axis" stroke-width="0.5" stroke-dasharray="3,3"/>
<text x="110" y="64" font-size="14" font-weight="500" class="heading" text-anchor="end">정책</text>
<text x="110" y="82" font-size="12" class="caption" text-anchor="end">20 ms · 1회</text>
<rect x="120" y="44" width="520" height="48" rx="6" class="pol-fill" stroke-width="0.5"/>
<text x="380" y="64" font-size="14" font-weight="500" class="pol-title" text-anchor="middle" dominant-baseline="central">q_des = π_θ(obs)</text>
<text x="380" y="82" font-size="12" class="pol-sub" text-anchor="middle" dominant-baseline="central">20 ms 동안 q_des 유지</text>
<text x="110" y="124" font-size="14" font-weight="500" class="heading" text-anchor="end">PD</text>
<text x="110" y="142" font-size="12" class="caption" text-anchor="end">5 ms · 4회</text>
<g class="teal-fill" stroke-width="0.5">
<rect x="120" y="104" width="127" height="48" rx="6"/>
<rect x="251" y="104" width="127" height="48" rx="6"/>
<rect x="382" y="104" width="127" height="48" rx="6"/>
<rect x="513" y="104" width="127" height="48" rx="6"/>
</g>
<g font-size="14" font-weight="500" class="teal-title" text-anchor="middle">
<text x="183.5" y="124" dominant-baseline="central">PD #1</text>
<text x="314.5" y="124" dominant-baseline="central">PD #2</text>
<text x="445.5" y="124" dominant-baseline="central">PD #3</text>
<text x="576.5" y="124" dominant-baseline="central">PD #4</text>
</g>
<g font-size="12" class="teal-sub" text-anchor="middle">
<text x="183.5" y="142" dominant-baseline="central">τ_des, I_des</text>
<text x="314.5" y="142" dominant-baseline="central">τ_des, I_des</text>
<text x="445.5" y="142" dominant-baseline="central">τ_des, I_des</text>
<text x="576.5" y="142" dominant-baseline="central">τ_des, I_des</text>
</g>
<text x="110" y="184" font-size="14" font-weight="500" class="heading" text-anchor="end">물리</text>
<text x="110" y="202" font-size="12" class="caption" text-anchor="end">0.1 ms · 200회</text>
<g class="coral-fill" stroke-width="0.5">
<rect x="120" y="164" width="127" height="48" rx="6"/>
<rect x="251" y="164" width="127" height="48" rx="6"/>
<rect x="382" y="164" width="127" height="48" rx="6"/>
<rect x="513" y="164" width="127" height="48" rx="6"/>
</g>
<g font-size="14" font-weight="500" class="coral-title" text-anchor="middle">
<text x="183.5" y="184" dominant-baseline="central">substep × 50</text>
<text x="314.5" y="184" dominant-baseline="central">substep × 50</text>
<text x="445.5" y="184" dominant-baseline="central">substep × 50</text>
<text x="576.5" y="184" dominant-baseline="central">substep × 50</text>
</g>
<g font-size="12" class="coral-sub" text-anchor="middle">
<text x="183.5" y="202" dominant-baseline="central">q, q̇, I 갱신</text>
<text x="314.5" y="202" dominant-baseline="central">q, q̇, I 갱신</text>
<text x="445.5" y="202" dominant-baseline="central">q, q̇, I 갱신</text>
<text x="576.5" y="202" dominant-baseline="central">q, q̇, I 갱신</text>
</g>
<text x="380" y="244" font-size="12" class="caption" text-anchor="middle">바깥 척도 산출값은 다음 갱신까지 상수로 유지 → 안쪽 척도가 그 값을 받아 더 자주 갱신</text>
<text x="380" y="262" font-size="12" class="caption" text-anchor="middle">마지막 substep 의 (q, q̇, I) 가 다음 정책 obs 의 일부가 된다</text>
</svg>


## 2. 정책 단계 (20 ms)

$$
a_k = \pi_\theta(o_k),
\qquad
q_{\mathrm{des},k} = q_{\mathrm{default}} + s_a\,a_k,
\qquad s_a = 0.25
$$

기호:
- $a_k\in\mathbb{R}^{12}$, $q_{\mathrm{des},k}\in\mathbb{R}^{12}$, $\pi_\theta$:
  4-layer ELU MLP `[512,256,128]` (`obs_normalizer` 거친 후), $o_k\in\mathbb{R}^{47}$
- $q_{\mathrm{default}}$: home 포즈 (hip=0, thigh=0.9, calf=−1.8) × 4 다리

관측 $o_k$ (47 dims) 구성: `[ang_vel(3), proj_g(3), cmd(3), phase(2),
joint_pos_rel(12), joint_vel_rel(12), last_action(12)]`. 12 관절 순서는 xml DFS
(`FL_h FL_t FL_c | FR_h FR_t FR_c | RL_… | RR_…`). 본 노트북은 FR (인덱스 3,4,5)
3 관절만 사용.

ckpt: `logs/rsl_rl/go2_methoda_electric/2026-04-27_23-25-13_act-pos_pdt20ms_phyDt0p1ms_tauDec4/model_1999.pt`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

# ── ckpt 로드 ──
CKPT_PATH = (
    "model_1999.pt"
)
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
sd = ckpt["actor_state_dict"]
obs_mean = sd["obs_normalizer._mean"][0]
obs_std  = sd["obs_normalizer._std"][0]

mlp = nn.Sequential(
    nn.Linear(47, 512), nn.ELU(),
    nn.Linear(512, 256), nn.ELU(),
    nn.Linear(256, 128), nn.ELU(),
    nn.Linear(128, 12),
)
for src_idx, dst in [(0, 0), (2, 2), (4, 4), (6, 6)]:
    mlp[dst].weight.data = sd[f"mlp.{src_idx}.weight"]
    mlp[dst].bias.data   = sd[f"mlp.{src_idx}.bias"]
mlp.eval()

# ── 관측 (이 값을 바꿔보세요) ──
ang_vel    = np.array([0.0, 0.0, 0.0])
proj_g     = np.array([0.0, 0.0, -1.0])
cmd_twist  = np.array([0.5, 0.0, 0.0])
phase      = np.array([0.0, 1.0])
q_default  = np.array([0.0, 0.9, -1.8] * 4)
q_full     = q_default.copy()
q_dot_full = np.zeros(12)
last_action = np.zeros(12)

obs = np.concatenate([
    ang_vel, proj_g, cmd_twist, phase,
    q_full - q_default, q_dot_full, last_action,
])
assert obs.shape == (47,)

# ── π_θ forward ──
obs_t = torch.from_numpy(obs).float()
obs_n = (obs_t - obs_mean) / obs_std
with torch.no_grad():
    action_raw = mlp(obs_n.unsqueeze(0))[0].numpy()

ACTION_SCALE = 0.25
q_des_full = q_default + ACTION_SCALE * action_raw
print(f"a_k        = {action_raw.round(5)}")
print(f"q_des      = {q_des_full.round(5)}")

# FR (3,4,5) 슬라이스 → 다음 절 입력
fr_idx = slice(3, 6)
q, q_dot, q_des = q_full[fr_idx], q_dot_full[fr_idx], q_des_full[fr_idx]
print(f"FR  q={q}, q_dot={q_dot}, q_des={q_des}")


## 3. PD 제어 단계 (5 ms)

PD 재계산 시각 $t_m$ 에 한 번 계산하고, 다음 PD 갱신 전까지 50 substep 동안
$I_{\mathrm{des}}$ 를 캐시 (ZOH).

$$
\tau_{\mathrm{des}} = K_p\,(q_{\mathrm{des}} - q) - K_d\,\dot q,
\qquad
I_{\mathrm{des}} = \frac{\tau_{\mathrm{des}}}{K_t\,g_r}
$$

기호:
- $K_p,\,K_d$: 관절별 stiffness/damping (HIP=20/1, THIGH=20/1, CALF=40/2)
- $K_t,\,g_r$: torque constant, gear ratio (Method A: $K_t=K_e=0.128$, $g_r=6.33$)


In [ ]:
# FR_hip — cell 위의 q_des, q, q_dot 를 그대로 받아 PD 계산
Kp, Kd       = 20.0, 1.0
Kt, gr       = 0.128, 6.33
Kt_gr        = Kt * gr

j = 0  # FR_hip
tau_des = Kp * (q_des[j] - q[j]) - Kd * q_dot[j]
I_des   = tau_des / Kt_gr
print(f"τ_des = {tau_des:.4f} [N·m]")
print(f"I_des = τ_des / (Kt·gr) = {I_des:.4f} [A]   (Kt·gr = {Kt_gr:.4f})")


## 4. 물리 단계 (0.1 ms)

### 4.1 운동방정식

원래 mjwarp 는 한 step 안에서 $n_{\mathrm{efc}}$ 차원 convex 최적화 (Gauss principle of
least constraint, `solver.py` PCG/Newton) 로 $J_c^\top\lambda$ 를 풀어 $\ddot q$ 를
보정한다. 본 노트는 직접 contact constraint 가 없는 hip 관절 (FR_hip) 만 다루므로
$J_c = 1$, $\lambda = 0$ 인 1자유도 모델로 단순화한다:

$$
M(q)\,\ddot q + b\,\dot q + C(q,\dot q) + g(q) \;=\; \tau_{\mathrm{actuator}},
\qquad \tau_{\mathrm{actuator}} = K_t\,g_r\,I
$$

기호:
- $M(q)$: mass matrix, $b$: joint damping, $C(q,\dot q)$: Coriolis 일반화 힘,
  $g(q)$: 중력 일반화 힘
- $\tau_{\mathrm{actuator}}$: actuator 가 관절에 가하는 토크
  (소스: `act.gainprm[0] = K_t g_r`, MuJoCo `force = gainprm[0] * act`)

비구속 가속도:

$$
\ddot q_{\mathrm{free}} = M^{-1}\bigl(\tau_{\mathrm{actuator}} - b\dot q - C - g\bigr)
$$


In [ ]:
import numpy as np

# 자유도 2 축소 모형 (이 값을 바꿔보세요)
M = np.array([[1.5, 0.1],
              [0.1, 0.8]])
q_dot         = np.array([0.20, -0.10])
damping_b     = np.array([0.30, 0.20])
damping_qdot  = damping_b * q_dot
C_qdot        = np.array([0.05, -0.02])
g_q           = np.array([3.00,  0.50])
tau_actuator  = np.array([4.0, 1.5])           # 이미 Kt·gr·I 형태

q_ddot_free   = np.linalg.inv(M) @ (tau_actuator - damping_qdot - C_qdot - g_q)
print(f"q̈_free = {q_ddot_free}")


### 4.2 β_imp — Schur complement / Force RHS

이 절이 노트의 핵심이다. **왜 $M_{\mathrm{eff}}$ 와 Force RHS 가 그런 모양으로
나오는가** 를 block matrix → Schur 소거 → 코드 대응 순서로 따라간다.
$K_{e,\mathrm{nom}} = K_{e,\mathrm{plant}}$ 가정 (mismatch 항 제거).

#### 물리적 block matrix

한 step $[t,\,t+h]$ 동안 좌변의 $\Delta\dot q$, $\Delta i$ 가 한꺼번에 결정된다고
(즉 step 종료 시점 양에 대한 force balance 를) 둔 형태:

$$
\begin{bmatrix}
\dfrac{M}{h} + B_{\mathrm{mech}} & -K_t g_r \\[1.2em]
K_e g_r & \dfrac{L}{h} + R
\end{bmatrix}
\begin{bmatrix}\Delta \dot q \\[0.4em] \Delta i\end{bmatrix}
=
\begin{bmatrix} F_{\mathrm{mech}} \\[0.4em] F_{\mathrm{elec}} \end{bmatrix}
$$

기호 (이후 절에서도 동일하게 사용):
- $M(q)$: mass matrix, $b$: joint damping, $c(q,\dot q)$: Coriolis matrix, $G(q)$: 중력
- $B_{\mathrm{mech}} \equiv b + c + (\partial c/\partial\dot q)\dot q$
- $K_t g_r$: 전류 → 토크, $K_e g_r$: back-EMF
- $L,\,R$: winding inductance, resistance, $V$: motor 인가전압
- $h$: 물리 적분 시간 간격 (`m.opt.timestep`)
- $\dot q_k,\,i_k$: 현재 step 의 관절속도·전류, $\Delta\dot q,\,\Delta i$: 한 step 증분

abstract 표기 $\bigl[\begin{smallmatrix}A & B \\ C & D\end{smallmatrix}\bigr]$ 와 일대일 대응:

$$
A = \tfrac{M}{h} + B_{\mathrm{mech}},\quad
B = -K_t g_r,\quad
C = K_e g_r,\quad
D = \tfrac{L}{h} + R
$$

블록 우변의 두 항을 이름붙여 둔다 (Schur 전개 내내 사용):

$$
F_{\mathrm{mech}} \;\equiv\; K_t g_r\,i_k \;-\; b\dot q \;-\; c\dot q \;-\; G,
\qquad
F_{\mathrm{elec}} \;\equiv\; V \;-\; R\,i_k \;-\; K_e g_r\,\dot q_k
$$

- $F_{\mathrm{mech}}$: step 시작 전류 $i_k$ 만으로 계산한 actuator torque
  ($K_t g_r\,i_k$) + 속도성/중력 일반화 힘. 좌변이 0 일 때 (모터 coupling 무시)
  $\Delta\dot q$ 를 만들어 내는 우변.
- $F_{\mathrm{elec}}$: 인가전압에서 저항강하·역기전력을 뺀 잉여전압. 0 이면
  $\Delta i = 0$, 양수면 전류 증가.

#### $\Delta i$ 를 제거하는 Schur complement 전개

(i) 두 번째 행에서 $\Delta i$ 풀기:

$$
\Delta i \;=\; \frac{1}{L/h + R}\Bigl[\,F_{\mathrm{elec}} \;-\; K_e g_r\,\Delta\dot q\,\Bigr]
$$

(ii) 첫 행에 대입:

$$
\Bigl(\tfrac{M}{h} + B_{\mathrm{mech}}\Bigr)\Delta\dot q
\;-\; K_t g_r\cdot\frac{F_{\mathrm{elec}} - K_e g_r\,\Delta\dot q}{L/h + R}
\;=\; F_{\mathrm{mech}}
$$

(iii) $\Delta\dot q$ 항을 좌변에 모으고 $F_{\mathrm{elec}}$ 항을 우변으로:

$$
\underbrace{\Bigl[\tfrac{M}{h} + B_{\mathrm{mech}}
+ \tfrac{K_t g_r\,K_e g_r}{L/h + R}\Bigr]}_{\text{Schur complement }A - BD^{-1}C}\Delta\dot q
\;=\; F_{\mathrm{mech}} \;+\; \underbrace{\tfrac{K_t g_r}{L/h + R}\,F_{\mathrm{elec}}}_{\text{Force RHS 보정의 원형}}
$$

좌변 마지막 항이 $M_{\mathrm{eff}}$ 의 motor coupling 보정으로,
우변 마지막 항이 Force RHS 보정으로 이어진다.
**같은 분모 $L/h + R$ 이 두 곳에 동시에 등장**하는 것이 이 절의 핵심이다.

#### `qDeriv` / `M_eff` 코드 표기와의 연결

코드는 $\Delta\dot q$ 가 아니라 $\ddot q = \Delta\dot q / h$ 를 직접 푼다. 위 식 LHS 에
$\Delta\dot q = h\ddot q$ 를 대입하면 $M_{\mathrm{eff}}\,\ddot q$ 형태가 된다 (RHS 는 변화 없음):

$$
M_{\mathrm{eff}} \equiv M + h\,B_{\mathrm{mech}} + h\cdot\frac{K_t g_r\,K_e g_r}{L/h + R}
$$

마지막 항을 $\tau_e = L/R$ 로 정리하면 $L/h + R = R(\tau_e + h)/h$ 이므로

$$
h\cdot\frac{K_t g_r\,K_e g_r}{L/h + R}
\;=\; h\cdot\underbrace{\tfrac{h}{\tau_e+h}}_{=\,1-\beta_{\mathrm{imp}}}\cdot\frac{K_t g_r\,K_e g_r}{R}
$$

이게 코드의 `schur` 부호반전 형태다 (`schur = -one_minus_beta * Kt_gr * Ke_gr / R`).
1자유도 ($J_c = 1$) 모델에서는

$$
\boxed{\;M_{\mathrm{eff}} \;=\; M \;+\; h\,(1-\beta_{\mathrm{imp}})\,\frac{K_t g_r\,K_e g_r}{R}\;}
$$

이 한 줄로 끝.

#### Force RHS 보정의 자세한 전개

위에서 LHS 만 $\Delta\dot q = h\ddot q$ 대입으로 $M_{\mathrm{eff}}\,\ddot q$ 가 됐고,
RHS 는 그대로 두 덩어리:

$$
\text{RHS} \;=\; F_{\mathrm{mech}}
\;+\; \underbrace{\tfrac{K_t g_r}{L/h + R}\,F_{\mathrm{elec}}}_{\text{Schur RHS 보정}}
$$

(i) Schur RHS 보정의 인수 $K_t g_r/(L/h + R)$ 를 $\tau_e$ 로 정리.
$L/h + R = R(\tau_e + h)/h$ 이므로

$$
\frac{K_t g_r}{L/h + R}
\;=\; \frac{h\,K_t g_r}{R(\tau_e + h)}
\;=\; \underbrace{\tfrac{h}{\tau_e + h}}_{=\,1-\beta_{\mathrm{imp}}}\cdot\frac{K_t g_r}{R}
$$

→ **Schur RHS 보정도 좌변 `schur` 와 같은 $(1-\beta_{\mathrm{imp}})$, 같은 $K_t g_r/R$**
이 공통으로 들어간다 ($K_e g_r$ 만 빠진 자리에 $F_{\mathrm{elec}}$ 이 들어옴).
$h$ 는 사라진 게 아니라 $1-\beta_{\mathrm{imp}} = h/(\tau_e + h)$ 안으로 흡수됐다.

(ii) 모터 컨트롤러 가정으로 $F_{\mathrm{elec}}$ 풀기. Method A (BE) 분기에서
전류 ODE 는 $dI/dt = (\mathrm{ctrl} - I)/\tau_e$ 이다. 이게 실제 1차 회로
$L\,dI/dt = V - R I - K_e g_r \dot q$ 와 같아지려면 컨트롤러가

$$
V \;=\; R\,\mathrm{ctrl} \;+\; K_e g_r\,\dot q_k
$$

로 인가전압을 잡아 줘야 한다 (저항강하 + 역기전력 보상 = ideal current servo).
이를 $F_{\mathrm{elec}}$ 정의에 대입하면

$$
F_{\mathrm{elec}}
\;=\; (R\,\mathrm{ctrl} + K_e g_r\,\dot q_k) - R\,i_k - K_e g_r\,\dot q_k
\;=\; R\,(I_{\mathrm{des}} - i_k)
$$

(iii) (i)·(ii) 합치면 Schur RHS 보정 항은

$$
(1-\beta_{\mathrm{imp}})\,\frac{K_t g_r}{R}\cdot R\,(I_{\mathrm{des}} - i_k)
\;=\; K_t g_r\,(1-\beta_{\mathrm{imp}})\,(I_{\mathrm{des}} - i_k)
$$

(iv) RHS 를 다시 쓰면

$$
\boxed{\;\text{RHS} \;=\; F_{\mathrm{mech}} \;+\; K_t g_r\,(1-\beta_{\mathrm{imp}})\,(I_{\mathrm{des}} - i_k)\;}
$$

(v) MuJoCo 코드 매칭. MuJoCo 는 actuator 토크와 bias force ($-b\dot q - c\dot q - G$) 를
별도 버퍼에 저장하므로, $F_{\mathrm{mech}}$ 의 actuator 토크 부분만 떼어
$F_{\mathrm{pre}} \equiv K_t g_r\,i_k$ 로 두면 implicit 보정된 actuator 토크
$F_{\mathrm{post}}$ 는

$$
F_{\mathrm{post}} \;=\; F_{\mathrm{pre}} \;+\; K_t g_r\,(1-\beta_{\mathrm{imp}})\,(I_{\mathrm{des}} - i_k)
\;=\; K_t g_r\bigl[\,\beta_{\mathrm{imp}}\,i_k + (1-\beta_{\mathrm{imp}})\,I_{\mathrm{des}}\,\bigr]
$$

즉 actuator force 의 인자를 "step 시작 전류 $i_k$" 가 아니라
"한 step 안에서 $\beta_{\mathrm{imp}}$ 와 $1-\beta_{\mathrm{imp}}$ 로 가중평균한
implicit 예측 전류" 로 바꾼 것이다.

#### mjwarp 코드와의 대응

| 본문 항 | 코드 |
|---------|------|
| $1-\beta_{\mathrm{imp}} = h/(\tau_e + h)$ | `one_minus_beta = h_dt / (tau_e + h_dt)` (`derivative.py:87`) |
| Schur scalar $-(1-\beta_{\mathrm{imp}})\,K_t g_r K_e g_r / R$ | `schur = -one_minus_beta * Kt_gr * Ke_gr / R_val` (`:89`) |
| $M_{\mathrm{eff}} = qM - h\cdot q\mathrm{Deriv}$ | `qM_in - qderiv` (after `qderiv *= h`) (`:209-217`) |
| $F_{\mathrm{pre}} = K_t g_r\,i_k$ | `force = gain * ctrl_act + bias` (`forward.py:765`) |
| $\Delta F = K_t g_r(1-\beta_{\mathrm{imp}})(I_{\mathrm{des}} - i_k)$ | `force += gain * one_minus_beta * (ctrl - I_old)` (`:787`) |

`gain = K_t g_r`, `ctrl = I_des`, `I_old = i_k`. 좌변 `schur` 와 우변 보정이
같은 $(1-\beta_{\mathrm{imp}})$ 인수를 공유하는 게 한 step 안에서 $i$ 와 $\dot q$ 를
일관되게 implicit 으로 묶는다 — 이 짝짓기가 본 노트북이 다루는 Method A (BE) 분기의 정의다.

In [ ]:
import numpy as np

# 모터·통합 파라미터 (이 값을 바꿔보세요)
Kt, Ke, gr = 0.128, 0.128, 6.33
R, L       = 0.3, 1e-4
h          = 1e-4

tau_e   = L / R
Kt_gr   = Kt * gr
Ke_gr   = Ke * gr

# β_imp (Method A: BE 분기)
one_minus_beta_imp = h / (tau_e + h)
beta_imp           = 1.0 - one_minus_beta_imp
print(f"τ_e = {tau_e*1e6:.2f} µs,  β_imp = {beta_imp:.6f},  1-β = {one_minus_beta_imp:.6f}")

# 1자유도 ($J_c = 1$): M_eff = M + h(1-β_imp)·Kt·gr·Ke·gr/R
M_scalar = 0.5
M_eff    = M_scalar + h * one_minus_beta_imp * Kt_gr * Ke_gr / R
print(f"M = {M_scalar:.6f},  M_eff = {M_eff:.6f}")

# Force RHS 보정: F_post = Kt·gr·I + Kt·gr·(1-β_imp)·(I_des - I)
I_old = 1.0
I_des = 5.0
F_pre  = Kt_gr * I_old
F_post = F_pre + Kt_gr * one_minus_beta_imp * (I_des - I_old)
print(f"F_pre = {F_pre:.6f},  F_post = {F_post:.6f}  [N·m]")
# 등가 형태:  F_post = Kt·gr·[β·I + (1-β)·I_des]
F_post_eq = Kt_gr * (beta_imp * I_old + one_minus_beta_imp * I_des)
print(f"F_post (가중평균 형태) = {F_post_eq:.6f}   ← F_post 와 일치해야 함")

# 비구속 가속도 (접촉 무시)
q_dot_scalar = 0.20
damping_b    = 0.30
C_q, g_q_v   = 0.0, 2.0
qddot = (F_post - damping_b * q_dot_scalar - C_q - g_q_v) / M_eff
print(f"q̈ = {qddot:.6f}")


### 4.3 β_int — 적분 단계

§4.2 의 Schur 풀이 결과 $\Delta\dot q$ 를 $\ddot q := \Delta\dot q/h$ 로 다시 쓴 것이
mjwarp 의 `qacc` 다 (정의상 $\Delta\dot q = h\ddot q$). 따라서

$$
\dot q_{n+1} \;=\; \dot q_n + \Delta\dot q \;=\; \dot q_n + h\,\ddot q_n
$$

는 forward Euler 가 아니라 **§4.2 의 implicit 한 step 을 형태만 바꿔 적은 것**이다.
"$\dot q + h\ddot q$" 의 모양은 같지만, 들어가는 $\ddot q$ 가 bare $M^{-1}(\tau - …)$ 이
아니라 Schur-corrected $M_{\mathrm{eff}}^{-1}(F_{\mathrm{post}} - …)$ 라서 BE 한 step 이
된다. 위치는 갱신된 속도를 써서 semi-implicit:

$$
\dot q_{n+1} = \dot q_n + h\ddot q_n,
\qquad
q_{n+1} = q_n + h\dot q_{n+1}
$$

#### $\dot I$ 가 어떻게 나오는가 (전기 회로 → $\tau_e$)

DC 모터 winding 의 KVL (Kirchhoff voltage law):

$$
L\,\frac{dI}{dt} \;=\; V \;-\; R\,I \;-\; K_e\,g_r\,\dot q
$$

기호:
- $V$: 인가전압, $R$: 권선 저항, $L$: 인덕턴스
- $L\,dI/dt$: 인덕터 전압강하, $R\,I$: 저항 전압강하, $K_e g_r\,\dot q$: back-EMF
  (관절속도가 만드는 역기전력)

§4.2 의 ideal current servo 컨트롤러 가정으로 인가전압을

$$
V \;=\; R\,I_{\mathrm{des}} \;+\; K_e\,g_r\,\dot q
$$

(저항강하 보상 + 역기전력 보상) 으로 잡아 주면, KVL 에 대입했을 때
$K_e g_r\,\dot q$ 두 항이 상쇄되고

$$
L\,\dot I \;=\; (R\,I_{\mathrm{des}} + K_e g_r\,\dot q) - R\,I - K_e g_r\,\dot q
\;=\; R\,(I_{\mathrm{des}} - I)
$$

만 남는다. 양변을 $L$ 로 나누고 전기 시정수 $\tau_e \equiv L/R$ 로 쓰면

$$
\dot I \;=\; \frac{R}{L}\,(I_{\mathrm{des}} - I) \;=\; \frac{I_{\mathrm{des}} - I}{\tau_e}
$$

step 시작 시점 $I = I_n$ 에서 평가하면 ($K_{e,\mathrm{nom}} = K_{e,\mathrm{plant}}$ 가정 하):

$$
\dot I_n \;=\; \frac{I_{\mathrm{des}} - I_n}{\tau_e}
$$

이게 mjwarp `_actuator_force` 의 `act_dot = (ctrl - act) / tau_e` 한 줄.

#### Method A 적분식

$$
I_{n+1} = I_n + \frac{h\,\dot I_n}{1 + h/\tau_e}
$$

(원본 코드: `act = act_in + act_dot_scale * act_dot_in * opt_timestep / (1.0 + opt_timestep/tau)`).

#### 가중평균 형태로 정리

(1) $\dot I_n$ 대입:

$$
I_{n+1} = I_n + \frac{h}{1 + h/\tau_e}\cdot\frac{I_{\mathrm{des}} - I_n}{\tau_e}
= I_n + \frac{h\,(I_{\mathrm{des}} - I_n)}{\tau_e\,(1 + h/\tau_e)}
$$

(2) 분모 정리, $\tau_e\,(1+h/\tau_e) = \tau_e + h$:

$$
I_{n+1} = I_n + \frac{h}{\tau_e + h}\,(I_{\mathrm{des}} - I_n)
$$

(3) $\beta_{\mathrm{int}} \equiv 1/(1+h/\tau_e) = \tau_e/(\tau_e+h)$ 정의 →
$1-\beta_{\mathrm{int}} = h/(\tau_e+h)$. (2) 의 계수가 정확히 $1-\beta_{\mathrm{int}}$:

$$
I_{n+1} = I_n + (1-\beta_{\mathrm{int}})\,(I_{\mathrm{des}} - I_n)
$$

(4) $I_n$ 항 모으기:

$$
I_{n+1} = \bigl[1 - (1-\beta_{\mathrm{int}})\bigr] I_n + (1-\beta_{\mathrm{int}})\,I_{\mathrm{des}}
$$

$$
\boxed{\;I_{n+1} = \beta_{\mathrm{int}}\,I_n + (1-\beta_{\mathrm{int}})\,I_{\mathrm{des}}\;}
$$

해석: $\beta_{\mathrm{int}}$ 와 $1-\beta_{\mathrm{int}}$ 의 합이 1 이므로 $I_{n+1}$ 은
"현재값 $I_n$" 과 "타겟 $I_{\mathrm{des}}$" 의 **가중평균**. $h \ll \tau_e$ 면
$\beta_{\mathrm{int}} \to 1$ 로 한 step 에 거의 안 움직이고, $h \gg \tau_e$ 면
$\beta_{\mathrm{int}} \to 0$ 으로 한 step 에 $I_{\mathrm{des}}$ 에 거의 다 도달한다.

코드 위치: `_advance` 가 `_next_velocity` (`qvel += h*qacc`) → `_next_position`
(`qpos += h*qvel_new`) 순서로 실행 (`forward.py:251-300`).


### 4.3.1 Demag (Ke mismatch) 일반화

위까지는 healthy 가정 $K_{e,\mathrm{nom}} = K_{e,\mathrm{plant}}$ 을 깔았다.
demagnetization 이 일어나면 plant 의 자속이 약해져 $K_{e,\mathrm{plant}} <
K_{e,\mathrm{nom}}$ 이 되고, 컨트롤러는 그 사실을 몰라 **nominal 값 $K_{e,\mathrm{nom}}$
으로 back-EMF 보상**을 계속한다. 그래서 §4.2 의 두 항이 더 이상 깔끔히 상쇄되지
않고 mismatch 가 남는다.

#### plant KVL vs 컨트롤러

$$
\text{plant:}\;\; L\,\dot I = V - R\,I - K_{e,\mathrm{plant}}\,g_r\,\dot q,
\qquad
\text{controller:}\;\; V = R\,I_{\mathrm{des}} + K_{e,\mathrm{nom}}\,g_r\,\dot q
$$

대입하면 $\dot q$ 항이 완전히 상쇄되지 않고

$$
L\,\dot I = R\,(I_{\mathrm{des}} - I) + (K_{e,\mathrm{nom}} - K_{e,\mathrm{plant}})\,g_r\,\dot q
$$

$L$ 로 나누고 $\tau_e = L/R$ 로 정리:

$$
\boxed{\;\dot I = \frac{I_{\mathrm{des}} - I}{\tau_e}
\;+\; \frac{(K_{e,\mathrm{nom}} - K_{e,\mathrm{plant}})\,g_r\,\dot q}{L}\;}
$$

마찬가지로 §4.2 의 $F_{\mathrm{elec}}$ 도 demag 항이 살아남는다:

$$
F_{\mathrm{elec}} = R\,(I_{\mathrm{des}} - I_n) + (K_{e,\mathrm{nom}} - K_{e,\mathrm{plant}})\,g_r\,\dot q_k
$$

→ Schur RHS 보정 $(1-\beta_{\mathrm{imp}})\,(K_t g_r / R)\,F_{\mathrm{elec}}$ 을 끝까지 풀면

$$
\boxed{\;\Delta F_{\mathrm{post}} =
\underbrace{K_t g_r\,(1-\beta_{\mathrm{imp}})\,(I_{\mathrm{des}} - I_n)}_{\text{vanilla}}
\;+\;
\underbrace{(1-\beta_{\mathrm{imp}})\,\frac{K_t g_r\,(K_{e,\mathrm{nom}} - K_{e,\mathrm{plant}})\,g_r\,\dot q_k}{R}}_{\text{demag piece}}\;}
$$

#### 세 단계에서의 영향 (코드 반영 후)

| 단계 | healthy ($K_{e,\mathrm{nom}} = K_{e,\mathrm{plant}}$) | demag ($K_{e,\mathrm{nom}} \neq K_{e,\mathrm{plant}}$) |
|------|---------------------------------------------|----------------------------------------------|
| Schur ($M_{\mathrm{eff}}$) | $M + h(1-\beta)\,\dfrac{K_t g_r\,K_e g_r}{R}$ | $M + h(1-\beta)\,\dfrac{K_t g_r\,K_{e,\mathrm{plant}}\,g_r}{R}$ — Schur scalar 가 plant $K_e$ 사용 |
| Force RHS 보정 | $K_t g_r\,(1-\beta)\,(I_{\mathrm{des}} - I_n)$ | 위 vanilla + **demag piece** $(1-\beta)\,K_t g_r\,(K_{e,\mathrm{nom}} - K_{e,\mathrm{plant}})\,g_r\,\dot q_k / R$ |
| 전류 적분 | $\dot I_n = (I_{\mathrm{des}} - I_n)/\tau_e$ → 가중평균식 | $\dot I_n$ 에 $(K_{e,\mathrm{nom}} - K_{e,\mathrm{plant}})\,g_r\,\dot q_k / L$ 가 추가된 뒤 동일한 Method A 적분식 |

세 곳 모두 demag 효과가 일관되게 들어가 §4.2 의 block matrix → Schur 유도가
끝까지 깨지지 않는다. healthy 케이스에서는 mismatch $(K_{e,\mathrm{nom}} -
K_{e,\mathrm{plant}}) = 0$ 으로 demag piece 가 그대로 0 이라 기존 동작과
bit-identical.

mismatch 항의 크기 감각: $h = 10^{-4}$, $\tau_e \approx 3.3\times 10^{-4}$ 로
$1-\beta \approx 0.23$ 일 때, 10% demag + $\dot q = 12$ rad/s 면 demag piece 는
vanilla 보정과 같은 차수 (둘 다 $\sim 0.6$–$0.8$ N·m) 가 나올 수 있다 —
무시할 수 없는 영역.


In [ ]:
import numpy as np

# 4.2 에서 받은 q, q̇, q̈ (1자유도)
q       = np.array([0.05])
q_dot   = np.array([0.10])
q_ddot  = np.array([qddot])
dt      = 1e-4

# 기계측 semi-implicit Euler
q_dot_new = q_dot + dt * q_ddot
q_new     = q     + dt * q_dot_new
print(f"q̇_new = {q_dot_new},  q_new = {q_new}")

# 전기측: Method A 전류 적분
Kt, Ke, gr = 0.128, 0.128, 6.33
R, L       = 0.3, 1e-4
tau_e      = L / R

I_n        = 1.0
I_des      = 5.0
act_dot    = (I_des - I_n) / tau_e
I_np1      = I_n + dt * act_dot / (1.0 + dt / tau_e)

beta_int   = 1.0 / (1.0 + dt / tau_e)
I_np1_eq   = beta_int * I_n + (1.0 - beta_int) * I_des
print(f"β_int = {beta_int:.6f},  I_{{n+1}} = {I_np1:.6f} (β형 = {I_np1_eq:.6f})")


### 4.4 한 substep 요약

화살표:
$\;I_{\mathrm{des}}, q, \dot q, I \to$ **EOM** $\to$ **β_imp (Schur·Force RHS)**
$\to$ **β_int (적분)** $\to$ 다음 $(q, \dot q, I)$.

압축 정리 (1자유도, $J_c = 1$):

$$
M_{\mathrm{eff}} = M + h\,(1-\beta_{\mathrm{imp}})\,\frac{K_t g_r\,K_e g_r}{R},
\qquad
F_{\mathrm{post}} = K_t g_r\,I + K_t g_r\,(1-\beta_{\mathrm{imp}})\,(I_{\mathrm{des}} - I)
$$

$$
\ddot q = M_{\mathrm{eff}}^{-1}(F_{\mathrm{post}} - b\dot q - C - g),
\quad
\dot q_{n+1} = \dot q_n + h\ddot q,
\quad
q_{n+1} = q_n + h\dot q_{n+1},
\quad
I_{n+1} = I_n + \frac{h(I_{\mathrm{des}} - I_n)/\tau_e}{1 + h/\tau_e}
$$

아래 셀은 위 식을 `one_substep` 로 묶어 1 PD 주기 (5 ms = 50 substep) 동안 반복.


In [ ]:
import numpy as np
import pandas as pd

def one_substep(q, q_dot, I, ctrl, M, damping_b, C_q, g_q_v, params):
    Kt, Ke, gr = params['Kt'], params['Ke'], params['gr']
    R,  L,  h  = params['R'],  params['L'],  params['h']
    Kt_gr, Ke_gr = Kt * gr, Ke * gr
    tau_e        = L / R

    # 4.2 β_imp
    omb    = h / (tau_e + h)
    M_eff  = M + h * omb * Kt_gr * Ke_gr / R
    F_post = Kt_gr * I + Kt_gr * omb * (ctrl - I)

    # 4.1 (접촉 무시)
    qddot = (F_post - damping_b * q_dot - C_q - g_q_v) / M_eff

    # 4.3 적분
    q_dot_new = q_dot + h * qddot
    q_new     = q     + h * q_dot_new
    I_new     = I + h * ((ctrl - I) / tau_e) / (1.0 + h / tau_e)
    return q_new, q_dot_new, I_new


params = dict(Kt=0.128, Ke=0.128, gr=6.33, R=0.3, L=1e-4, h=1e-4)
ctrl = 5.0
M, damping_b, C_q, g_q_v = 0.5, 0.30, 0.0, 0.0
q, q_dot, I = 0.0, 0.0, 0.0

N_SUBSTEPS_PER_PD = 50
rows = [{"k": 0, "q": q, "q_dot": q_dot, "I": I}]
for k in range(1, N_SUBSTEPS_PER_PD + 1):
    q, q_dot, I = one_substep(q, q_dot, I, ctrl, M, damping_b, C_q, g_q_v, params)
    rows.append({"k": k, "q": q, "q_dot": q_dot, "I": I})

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda x: f"{x: .6f}"))


## 5. 전체 흐름도

<svg id="fc-s5" width="100%" viewBox="0 0 680 644" xmlns="http://www.w3.org/2000/svg" style="max-width: 720px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">
<style>
#fc-s5 text{fill:#FFFFFF}
#fc-s5 .arrow{stroke:#FFFFFF;fill:none}
#fc-s5 .arrow-loop{stroke:#FFFFFF;fill:none;stroke-dasharray:4,3}
#fc-s5 .con-pol{fill:none;stroke:#AFA9EC}#fc-s5 .box-pol{fill:#3C3489;stroke:#AFA9EC}
#fc-s5 .con-pd{fill:none;stroke:#5DCAA5}#fc-s5 .box-pd{fill:#085041;stroke:#5DCAA5}
#fc-s5 .con-phy{fill:none;stroke:#F0997B}#fc-s5 .box-phy{fill:#712B13;stroke:#F0997B}
</style>
<defs>
<marker id="arr5" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
<path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
</marker>
</defs>
<rect class="con-pol" x="20" y="20" width="640" height="110" rx="8" stroke-width="0.5" stroke-dasharray="4,3"/>
<text class="lbl-pol" x="40" y="42" font-size="13" font-weight="500">정책 (20 ms)</text>
<rect class="box-pol" x="60" y="56" width="560" height="64" rx="6" stroke-width="0.5"/>
<text class="t-pol" x="340" y="78" font-size="14" font-weight="500" text-anchor="middle" dominant-baseline="central">q_des = q_def + 0.25·a    (a = π_θ(obs))</text>
<text class="ts-pol" x="340" y="100" font-size="12" text-anchor="middle" dominant-baseline="central">obs (47): ω_b, g_proj, v_cmd, phase, q−q_def, q̇, last_a</text>
<line class="arrow" x1="340" y1="130" x2="340" y2="160" stroke-width="1.5" marker-end="url(#arr5)"/>
<text class="caption" x="350" y="148" font-size="12" dominant-baseline="central">q_des</text>
<rect class="con-pd" x="20" y="168" width="640" height="110" rx="8" stroke-width="0.5" stroke-dasharray="4,3"/>
<text class="lbl-pd" x="40" y="190" font-size="13" font-weight="500">PD (5 ms × 4)</text>
<rect class="box-pd" x="60" y="204" width="560" height="64" rx="6" stroke-width="0.5"/>
<text class="t-pd" x="340" y="226" font-size="14" font-weight="500" text-anchor="middle" dominant-baseline="central">τ_des = Kp·(q_des − q) − Kd·q̇</text>
<text class="ts-pd" x="340" y="248" font-size="12" text-anchor="middle" dominant-baseline="central">I_des = τ_des / (Kt·gr)</text>
<line class="arrow" x1="340" y1="278" x2="340" y2="308" stroke-width="1.5" marker-end="url(#arr5)"/>
<text class="caption" x="350" y="296" font-size="12" dominant-baseline="central">I_des</text>
<rect class="con-phy" x="20" y="316" width="640" height="290" rx="8" stroke-width="0.5" stroke-dasharray="4,3"/>
<text class="lbl-phy" x="40" y="338" font-size="13" font-weight="500">물리 (0.1 ms × 50 per PD)</text>
<rect class="box-phy" x="60" y="352" width="520" height="56" rx="6" stroke-width="0.5"/>
<text class="t-phy" x="320" y="370" font-size="13" font-weight="500" text-anchor="middle" dominant-baseline="central">운동방정식 (접촉 무시, J_c=1)</text>
<text class="ts-phy" x="320" y="391" font-size="12" text-anchor="middle" dominant-baseline="central">M·q̈ + b·q̇ + C(q,q̇) + g(q) = Kt·gr·I</text>
<line class="arrow" x1="320" y1="408" x2="320" y2="420" stroke-width="1.5" marker-end="url(#arr5)"/>
<rect class="box-phy" x="60" y="422" width="520" height="80" rx="6" stroke-width="0.5"/>
<text class="t-phy" x="320" y="441" font-size="13" font-weight="500" text-anchor="middle" dominant-baseline="central">β_imp 보정 (Schur complement / Force RHS)</text>
<text class="ts-phy" x="320" y="463" font-size="12" text-anchor="middle" dominant-baseline="central">M_eff = M + h(1−β)·Kt·gr·Ke·gr/R</text>
<text class="ts-phy" x="320" y="484" font-size="12" text-anchor="middle" dominant-baseline="central">F_post = Kt·gr·I + Kt·gr·(1−β)·(I_des − I)</text>
<line class="arrow" x1="320" y1="502" x2="320" y2="514" stroke-width="1.5" marker-end="url(#arr5)"/>
<rect class="box-phy" x="60" y="516" width="520" height="80" rx="6" stroke-width="0.5"/>
<text class="t-phy" x="320" y="535" font-size="13" font-weight="500" text-anchor="middle" dominant-baseline="central">β_int 적분</text>
<text class="ts-phy" x="320" y="557" font-size="12" text-anchor="middle" dominant-baseline="central">I_{n+1} = I_n + h·(I_des − I_n)/τ_e / (1 + h/τ_e)</text>
<text class="ts-phy" x="320" y="578" font-size="12" text-anchor="middle" dominant-baseline="central">q̇_{n+1} = q̇_n + h·q̈,    q_{n+1} = q_n + h·q̇_{n+1}</text>
<path class="arrow-loop" d="M 580 556 L 615 556 L 615 380 L 580 380" stroke-width="1" marker-end="url(#arr5)"/>
<text class="caption" x="623" y="468" font-size="11" dominant-baseline="central">× 50</text>
<text class="caption" x="340" y="628" font-size="12" text-anchor="middle">마지막 substep 의 (q, q̇, I) → 다음 정책 주기 obs 의 일부 (전체 루프, 정책당 200 substep)</text>
</svg>

## 6. 한 정책 주기 누적 호출 횟수

| 단계 | 주기 | 정책 한 주기당 호출 횟수 |
|------|------|--------------------------|
| 정책 | 20 ms  | 1 |
| PD   | 5 ms   | 4   (= 20 ms / 5 ms) |
| 물리 | 0.1 ms | 200 (= 20 ms / 0.1 ms = `decimation`) |

PD 1 회당 물리 substep 수 = 50 (= 5 ms / 0.1 ms = `pd_substeps`).


## 7. 구현 위치 참조표

| 단계 | 본문 기호 | 코드상 명칭 | 파일 | 라인 |
|------|----------|-----------|------|------|
| 정책 | task 등록 | `register_mjlab_task("Unitree-Go2-Flat-MethodA-Electric", ...)` | `src/tasks/velocity/config/go2/__init__.py` | 81-93 |
| 정책 | 시간 설정 | `cfg.sim.mujoco.timestep`, `cfg.decimation` | `src/tasks/velocity/config/go2/env_cfgs.py` | 206-207 |
| PD   | substep 상수 | `_COUPLED_SUBSTEPS = 200`, `_PD_RECOMPUTE = 50` | `src/assets/robots/unitree_go2/go2_constants.py` | 305-306 |
| PD   | actuator cfg | `_MA_MOTOR`, `GO2_METHODA_HIP/THIGH/CALF` | `src/assets/robots/unitree_go2/go2_constants.py` | 342-358 |
| PD   | $\tau_{\mathrm{des}}$ | `tau_des = super().compute(cmd)` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 509 |
| PD   | $I_{\mathrm{des}}$    | `I_des = tau_des / self._Ktgr` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 531 |
| PD   | $I_{\mathrm{des}}(t) = I_{\mathrm{des}}(t_m)$ ZOH | `_I_des_hold`, `pd_substeps` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 497-540 |
| 물리 | $\tau_{\mathrm{actuator}} = K_t g_r I$ | `act.gainprm[0] = self._Ktgr` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 351-353 |
| 물리 | $1-\beta_{\mathrm{imp}}$ (Schur) | `one_minus_beta = h_dt / (tau_e + h_dt)` | `mujoco_warp/_src/derivative.py` | 87 |
| 물리 | Schur scalar | `schur = -one_minus_beta * Kt_gr * Ke_gr / R_val` | `mujoco_warp/_src/derivative.py` | 89 |
| 물리 | $M_{\mathrm{eff}} = qM - h\cdot q\mathrm{Deriv}$ | `qM_in - qderiv` (after `*= h`) | `mujoco_warp/_src/derivative.py` | 209-217 |
| 물리 | Force RHS 보정 (vanilla) | `force += gain * one_minus_beta * (ctrl - I_old)` | `mujoco_warp/_src/forward.py` | 787 |
| 물리 | Force RHS 보정 (demag piece) | `force += gain * one_minus_beta * ΔKe·gr · omega / R` | `mujoco_warp/_src/forward.py` | 793-796 |
| 물리 | $\dot I$ (filterexact) | `act_dot = (ctrl - act) / tau_e (+ ΔKe·gr·omega / L)` | `mujoco_warp/_src/forward.py` | 692-704 |
| 물리 | $\beta_{\mathrm{int}}$ 적분 | `act = act_in + ... * h / (1 + h/τ)` | `mujoco_warp/_src/forward.py` | 163-164 |
| 물리 | semi-implicit Euler | `_advance` (`_next_velocity` → `_next_position`) | `mujoco_warp/_src/forward.py` | 251-300 |
